# Domain-adaptive pretraining - NTSB narratives

Continues DistilBERT's masked-word pretraining on NTSB narratives (no labels), so the model learns aviation language before fine-tuning. Narratives from the test set of `finetune_transformer.ipynb` are excluded. The adapted model is then used by `finetune_transformer.ipynb` with `USE_DOMAIN_ADAPTED_MODEL = True`.

## Setup

In [1]:
import os
import math
import time

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from transformers import (AutoTokenizer, AutoModelForMaskedLM,
                          DataCollatorForLanguageModeling, get_linear_schedule_with_warmup)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


`PRETRAINING_SAMPLE_SIZE` and `EPOCHS` set the training time. 40,000 narratives x 2 epochs should take about 25 minutes on a T4; the progress lines show the estimated total.

In [3]:
BASE_MODEL_NAME = "distilbert-base-uncased"
MAXIMUM_TOKENS = 256
BATCH_SIZE = 32
PRETRAINING_SAMPLE_SIZE = 40000
EVALUATION_SAMPLE_SIZE = 1000
EPOCHS = 3
LEARNING_RATE = 5e-5
MASK_PROBABILITY = 0.15
RANDOM_STATE = 42

DRIVE_FOLDER = "/content/drive/MyDrive/aviation_finetune"
CLEAN_CASES_PATH = DRIVE_FOLDER + "/clean_cases.json"
ALL_NARRATIVES_PATH = DRIVE_FOLDER + "/all_narratives.json"
ADAPTED_MODEL_FOLDER = DRIVE_FOLDER + "/distilbert_ntsb_adapted"

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

## Google Drive

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## Load narratives

Put `clean_cases.json` and `all_narratives.json` (both created by `data_cleaning.ipynb`) in `MyDrive/aviation_finetune`.

In [5]:
cases_table = pd.read_json(CLEAN_CASES_PATH)
narratives_table = pd.read_json(ALL_NARRATIVES_PATH)

In [6]:
print("Labelled cases:", len(cases_table))
print("All narratives:", len(narratives_table))

Labelled cases: 21651
All narratives: 83172


## Exclude the test set

Same split as in `finetune_transformer.ipynb`, so the model never sees a test narrative before evaluation.

In [7]:
train_and_validation, test_table = train_test_split(
    cases_table, test_size=0.20, random_state=RANDOM_STATE, stratify=cases_table["label"])
narratives_table = narratives_table[~narratives_table["mKey"].isin(test_table["mKey"])]

In [8]:
print("Narratives after excluding the test set:", len(narratives_table))

Narratives after excluding the test set: 78843


## Sample

A small held-out sample measures how well the model guesses masked words (perplexity, lower is better).

In [9]:
shuffled_narratives = narratives_table["analysisNarrative"].sample(frac=1, random_state=RANDOM_STATE)
evaluation_texts = shuffled_narratives.iloc[:EVALUATION_SAMPLE_SIZE]
training_texts = shuffled_narratives.iloc[EVALUATION_SAMPLE_SIZE:EVALUATION_SAMPLE_SIZE + PRETRAINING_SAMPLE_SIZE]

In [10]:
print("Training narratives:", len(training_texts))
print("Evaluation narratives:", len(evaluation_texts))

Training narratives: 40000
Evaluation narratives: 1000


## Tokenization

In [11]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
class TextDataset(Dataset):
    def __init__(self, texts):
        self.encodings = tokenizer(list(texts), truncation=True, max_length=MAXIMUM_TOKENS)

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, index):
        return {key: values[index] for key, values in self.encodings.items()}

In [13]:
training_dataset = TextDataset(training_texts)
evaluation_dataset = TextDataset(evaluation_texts)

The collator hides 15% of the words; the model learns to guess them. The evaluation batches are masked once, so every evaluation uses the same hidden words.

In [14]:
collator = DataCollatorForLanguageModeling(tokenizer, mlm_probability=MASK_PROBABILITY)
training_loader = DataLoader(training_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator,
                             generator=torch.Generator().manual_seed(RANDOM_STATE))
evaluation_batches = list(DataLoader(evaluation_dataset, batch_size=BATCH_SIZE, collate_fn=collator))

## Model

In [15]:
model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL_NAME).to(device)

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [16]:
total_steps = len(training_loader) * EPOCHS
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.06 * total_steps),
                                            num_training_steps=total_steps)
use_mixed_precision = device.type == "cuda"
gradient_scaler = torch.amp.GradScaler("cuda", enabled=use_mixed_precision)

In [17]:
def evaluation_perplexity():
    model.eval()
    losses = []
    with torch.no_grad():
        for batch in evaluation_batches:
            batch = {key: values.to(device) for key, values in batch.items()}
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_mixed_precision):
                losses.append(model(**batch).loss.item())
    return math.exp(np.mean(losses))

In [18]:
perplexity_before = evaluation_perplexity()
print("Perplexity before pretraining:", round(perplexity_before, 2))

Perplexity before pretraining: 13.36


## Training

The adapted model is saved to Drive after every epoch.

In [19]:
history = []
training_start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_start_time = time.time()
    running_loss = 0
    for step, batch in enumerate(training_loader, start=1):
        batch = {key: values.to(device) for key, values in batch.items()}

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_mixed_precision):
            loss = model(**batch).loss

        optimizer.zero_grad()
        gradient_scaler.scale(loss).backward()
        gradient_scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        gradient_scaler.step(optimizer)
        gradient_scaler.update()
        scheduler.step()

        running_loss += loss.item()
        if step % 200 == 0:
            finished_steps = (epoch - 1) * len(training_loader) + step
            elapsed_minutes = (time.time() - training_start_time) / 60
            estimated_total_minutes = elapsed_minutes / finished_steps * total_steps
            print(f"Epoch {epoch} | step {step}/{len(training_loader)} | loss {running_loss / step:.3f} | "
                  f"{elapsed_minutes:.1f} of about {estimated_total_minutes:.0f} minutes")

    perplexity = evaluation_perplexity()
    history.append({"epoch": epoch, "train loss": running_loss / len(training_loader),
                    "evaluation perplexity": perplexity, "minutes": (time.time() - epoch_start_time) / 60})
    print(history[-1])

    model.save_pretrained(ADAPTED_MODEL_FOLDER)
    tokenizer.save_pretrained(ADAPTED_MODEL_FOLDER)
    print("Saved adapted model to", ADAPTED_MODEL_FOLDER)

/tmp/ipykernel_419/3636781221.py:20: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()


Epoch 1 | step 200/1250 | loss 2.091 | 0.9 of about 17 minutes
Epoch 1 | step 400/1250 | loss 1.911 | 1.8 of about 17 minutes
Epoch 1 | step 600/1250 | loss 1.806 | 2.8 of about 17 minutes
Epoch 1 | step 800/1250 | loss 1.735 | 3.7 of about 18 minutes
Epoch 1 | step 1000/1250 | loss 1.685 | 4.7 of about 18 minutes
Epoch 1 | step 1200/1250 | loss 1.642 | 5.7 of about 18 minutes
{'epoch': 1, 'train loss': 1.632931214618683, 'evaluation perplexity': 3.7824409190562123, 'minutes': 6.0454436858495075}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved adapted model to /content/drive/MyDrive/aviation_finetune/distilbert_ntsb_adapted
Epoch 2 | step 200/1250 | loss 1.387 | 7.1 of about 18 minutes
Epoch 2 | step 400/1250 | loss 1.374 | 8.1 of about 18 minutes
Epoch 2 | step 600/1250 | loss 1.365 | 9.1 of about 18 minutes
Epoch 2 | step 800/1250 | loss 1.357 | 10.1 of about 19 minutes
Epoch 2 | step 1000/1250 | loss 1.348 | 11.2 of about 19 minutes
Epoch 2 | step 1200/1250 | loss 1.338 | 12.2 of about 19 minutes
{'epoch': 2, 'train loss': 1.3350988221645355, 'evaluation perplexity': 3.3825112150586616, 'minutes': 6.384991804758708}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved adapted model to /content/drive/MyDrive/aviation_finetune/distilbert_ntsb_adapted
Epoch 3 | step 200/1250 | loss 1.261 | 13.6 of about 19 minutes
Epoch 3 | step 400/1250 | loss 1.259 | 14.6 of about 19 minutes
Epoch 3 | step 600/1250 | loss 1.255 | 15.6 of about 19 minutes
Epoch 3 | step 800/1250 | loss 1.250 | 16.7 of about 19 minutes
Epoch 3 | step 1000/1250 | loss 1.246 | 17.7 of about 19 minutes
Epoch 3 | step 1200/1250 | loss 1.244 | 18.7 of about 19 minutes
{'epoch': 3, 'train loss': 1.2443033543109894, 'evaluation perplexity': 3.2605412310973567, 'minutes': 6.408660348256429}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved adapted model to /content/drive/MyDrive/aviation_finetune/distilbert_ntsb_adapted


## Results

In [20]:
pd.DataFrame(history).round(3)

,epoch,train loss,evaluation perplexity,minutes
0,1,1.633,3.782,6.045
1,2,1.335,3.383,6.385
2,3,1.244,3.261,6.409


In [21]:
print("Perplexity before:", round(perplexity_before, 2))
print("Perplexity after :", round(history[-1]["evaluation perplexity"], 2))

Perplexity before: 13.36
Perplexity after : 3.26
